In [1]:
from google.colab import files

print("Upload your Lead Scoring CSV file now")
uploaded = files.upload()

Upload your Lead Scoring CSV file now


Saving Lead Scoring.csv to Lead Scoring.csv


In [2]:
import pandas as pd

# Load the data
df = pd.read_csv('Lead Scoring.csv')

# Basic exploration
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nTarget variable distribution:")
print(df['Converted'].value_counts())
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

Shape: (9240, 37)

Column names:
['Prospect ID', 'Lead Number', 'Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call', 'Converted', 'TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit', 'Last Activity', 'Country', 'Specialization', 'How did you hear about X Education', 'What is your current occupation', 'What matters most to you in choosing a course', 'Search', 'Magazine', 'Newspaper Article', 'X Education Forums', 'Newspaper', 'Digital Advertisement', 'Through Recommendations', 'Receive More Updates About Our Courses', 'Tags', 'Lead Quality', 'Update me on Supply Chain Content', 'Get updates on DM Content', 'Lead Profile', 'City', 'Asymmetrique Activity Index', 'Asymmetrique Profile Index', 'Asymmetrique Activity Score', 'Asymmetrique Profile Score', 'I agree to pay the amount through cheque', 'A free copy of Mastering The Interview', 'Last Notable Activity']

Target variable distribution:
Converted
0    5679
1    3561
Name: count, dtype: int64

Missing values pe

In [3]:
# Drop columns that are not useful
cols_to_drop = [
    'Prospect ID', 'Lead Number',  # ID columns - not useful for ML
    'Lead Quality', 'Tags',         # Too many missing values
    'Asymmetrique Activity Index', 'Asymmetrique Profile Index',
    'Asymmetrique Activity Score', 'Asymmetrique Profile Score',  # 46% missing
    'What matters most to you in choosing a course',
    'Lead Profile',                 # Too many missing values
    'Search', 'Magazine', 'Newspaper Article', 'X Education Forums',
    'Newspaper', 'Digital Advertisement', 'Through Recommendations',
    'Receive More Updates About Our Courses', 'Update me on Supply Chain Content',
    'Get updates on DM Content', 'I agree to pay the amount through cheque',
    'A free copy of Mastering The Interview'  # Low variance binary columns
]

df_clean = df.drop(columns=cols_to_drop)
print("Shape after dropping columns:", df_clean.shape)

# Impute categorical columns with mode
cat_cols_to_impute = [
    'What is your current occupation',
    'Country',
    'How did you hear about X Education',
    'Specialization',
    'City'
]

for col in cat_cols_to_impute:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Impute numerical columns with median
num_cols_to_impute = ['TotalVisits', 'Page Views Per Visit']

for col in num_cols_to_impute:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Replace 'Select' values with NaN then impute with mode
# (Select = user did not choose anything = essentially missing)
df_clean = df_clean.replace('Select', pd.NA)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Check missing values now
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum().sum())
print("\nShape after cleaning:", df_clean.shape)
print("\nRemaining columns:")
print(df_clean.columns.tolist())

Shape after dropping columns: (9240, 15)

Missing values after cleaning:
0

Shape after cleaning: (9240, 15)

Remaining columns:
['Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call', 'Converted', 'TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit', 'Last Activity', 'Country', 'Specialization', 'How did you hear about X Education', 'What is your current occupation', 'City', 'Last Notable Activity']


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Preserve lead identifier BEFORE dropping it
# We kept Prospect ID in df (original) but dropped it in df_clean
# So we restore it from the original df
lead_ids = df['Prospect ID'].reset_index(drop=True)

# Separate features and target
X = df_clean.drop(columns=['Converted'])
y = df_clean['Converted']

# One-hot encode categorical columns
X_encoded = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X_encoded.shape)

# Train/test split FIRST - before any scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set size:", X_train.shape)
print("Test set size:", X_test.shape)

# Fit scaler ONLY on training data then transform both sets separately
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

print("\nClass distribution in training set:")
print(y_train.value_counts())
print("\nScaler fitted on training data only ✅")

Shape after encoding: (9240, 132)

Training set size: (7392, 132)
Test set size: (1848, 132)

Class distribution in training set:
Converted
0    4543
1    2849
Name: count, dtype: int64

Scaler fitted on training data only ✅


In [9]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Training set after SMOTE:")
print("Class distribution:", np.bincount(y_train_smote))
print("Shape:", X_train_smote.shape)

# Train Random Forest - UNCHANGED
print("\nTraining Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_smote, y_train_smote)
print("Training complete!")

# Evaluate on test set - UNCHANGED
y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("\n--- MODEL EVALUATION ---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("AUC-ROC Score:", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training set after SMOTE:
Class distribution: [4543 4543]
Shape: (9086, 132)

Training Random Forest model...
Training complete!

--- MODEL EVALUATION ---

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.78      0.82      1136
           1       0.70      0.79      0.74       712

    accuracy                           0.79      1848
   macro avg       0.78      0.79      0.78      1848
weighted avg       0.80      0.79      0.79      1848

AUC-ROC Score: 0.8761

Confusion Matrix:
[[888 248]
 [146 566]]


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Tuned Random Forest - UNCHANGED
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train_smote, y_train_smote)

# Evaluate - UNCHANGED
y_pred_tuned = rf_tuned.predict(X_test_scaled)
y_prob_tuned = rf_tuned.predict_proba(X_test_scaled)[:, 1]

print("--- TUNED MODEL EVALUATION ---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))
print("AUC-ROC Score:", round(roc_auc_score(y_test, y_prob_tuned), 4))

--- TUNED MODEL EVALUATION ---

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.81      0.83      1136
           1       0.72      0.79      0.75       712

    accuracy                           0.80      1848
   macro avg       0.79      0.80      0.79      1848
weighted avg       0.81      0.80      0.80      1848

AUC-ROC Score: 0.8789


In [11]:
import joblib
import pandas as pd
from google.colab import files

# Scale ALL 9240 leads using scaler fitted on training data only
X_all_scaled = pd.DataFrame(
    scaler.transform(X_encoded),
    columns=X_encoded.columns
)

# Generate conversion probabilities for ALL leads
all_probs = rf_tuned.predict_proba(X_all_scaled)[:, 1]

# Assign priority tiers - UNCHANGED
def assign_tier(prob):
    if prob >= 0.70:
        return 'High'
    elif prob >= 0.40:
        return 'Medium'
    else:
        return 'Low'

# Create results dataframe WITH lead identifier restored
results_df = df_clean.copy()
results_df['prospect_id'] = lead_ids.values
results_df['conversion_probability'] = all_probs
results_df['priority_tier'] = results_df['conversion_probability'].apply(assign_tier)

# Reorder columns so prospect_id is first
cols = ['prospect_id'] + [
    col for col in results_df.columns if col != 'prospect_id'
]
results_df = results_df[cols]

# Check tier distribution - UNCHANGED
print("Priority Tier Distribution:")
print(results_df['priority_tier'].value_counts())

print("\nSample of results with lead identifier:")
print(results_df[[
    'prospect_id',
    'conversion_probability',
    'priority_tier',
    'Converted'
]].head(10))

# Verify prospect_id is present
print("\nProspect ID restored:", 'prospect_id' in results_df.columns)
print("Total leads scored:", len(results_df))

# Save all files
results_df.to_csv('leads_with_predictions.csv', index=False)
df_clean.to_csv('leads_cleaned.csv', index=False)
joblib.dump(rf_tuned, 'random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("\nAll files saved successfully!")

# Download all files
files.download('leads_with_predictions.csv')
files.download('leads_cleaned.csv')
files.download('random_forest_model.pkl')
files.download('scaler.pkl')

Priority Tier Distribution:
priority_tier
Low       4882
High      2205
Medium    2153
Name: count, dtype: int64

Sample of results with lead identifier:
                            prospect_id  conversion_probability priority_tier  \
0  7927b2df-8bba-4d29-b9a2-b6e0beafe620                0.129197           Low   
1  2a272436-5132-4136-86fa-dcc88c88f482                0.492074        Medium   
2  8cc8c611-a219-4f35-ad23-fdfd2656bd8a                0.570368        Medium   
3  0cc2df48-7cf4-4e39-9de9-19797f9b38cc                0.143937           Low   
4  3256f628-e534-4826-9d63-4a8b88782852                0.463139        Medium   
5  2058ef08-2858-443e-a01f-a9237db2f5ce                0.062337           Low   
6  9fae7df4-169d-489b-afe4-0f3d752542ed                0.593291        Medium   
7  20ef72a2-fb3b-45e0-924e-551c5fa59095                0.062337           Low   
8  cfa0128c-a0da-4656-9d47-0aa4e67bf690                0.187783           Low   
9  af465dfc-7204-4130-9e05-33231863c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>